# Aim 4 (Secondary): Outcomes Associated with Psychiatric Comorbidity in Epilepsy Surgery

**Project:** Prevalence and Trends of DSM-5 Axis I Psychiatric Disorders in Epilepsy Surgery  
**Data:** Analytic cohort from `01_cohort_building.ipynb`

This notebook (secondary aim):
1. Among epilepsy surgery patients: compare outcomes by psychiatric comorbidity status
2. Outcomes: in-hospital mortality, LOS, total charges, nonroutine discharge
3. Survey-weighted multivariable regression models
4. Subgroup analyses by disorder type

In [ ]:
# ============================================================
# SETUP
# ============================================================
library(arrow)
library(data.table)
library(survey)
library(ggplot2)
library(broom)

options(survey.lonely.psu = "adjust")

outdir <- "/Volumes/Niels 2/NIS_new_version/NIS_epilepsy_psych"

cat("Loading analytic cohort...\n")
dt <- as.data.table(read_parquet(file.path(outdir, "output/epilepsy_analytic.parquet")))

# Prepare variables
dt[, group := fifelse(any_surgery, "Surgery", "Non-Surgical")]
psych_cols <- grep("^psych_", names(dt), value = TRUE)
for (col in c(psych_cols, "any_psych")) dt[[col]] <- as.integer(dt[[col]])

# Outcome variables
dt[, nonroutine_dc := as.integer(DISPUNIFORM != 1)]  # 1 = routine
dt[, log_los := log(pmax(LOS, 1))]
dt[, log_totchg := log(pmax(TOTCHG, 1))]
dt[, died := as.integer(DIED == 1)]

# Demographics for adjustment
dt[, female := as.integer(FEMALE == 1)]
dt[, race_f := factor(RACE, levels = 1:6)]
dt[, pay_f := factor(PAY1, levels = 1:6)]

# SURGERY SUBSET
dt_surg <- dt[any_surgery == TRUE]
cat(sprintf("Surgery cohort: %s\n", format(nrow(dt_surg), big.mark = ",")))

# Survey designs
svy_all <- svydesign(id = ~HOSP_NIS, strata = ~NIS_STRATUM, weights = ~DISCWT, nest = TRUE, data = dt)
svy_surg <- svydesign(id = ~HOSP_NIS, strata = ~NIS_STRATUM, weights = ~DISCWT, nest = TRUE, data = dt_surg)

invisible(NULL)

## Unadjusted Outcomes: Surgery Patients by Psychiatric Status

In [ ]:
# ============================================================
# UNADJUSTED OUTCOMES: Surgery patients by psychiatric status
# ============================================================
cat("UNADJUSTED OUTCOMES (surgery patients only, survey-weighted)\n\n")

svy_surg_psych <- subset(svy_surg, any_psych == 1)
svy_surg_nopsych <- subset(svy_surg, any_psych == 0)

outcomes <- c("died", "nonroutine_dc", "LOS", "TOTCHG")
outcome_labels <- c("In-hospital mortality", "Nonroutine discharge", "Length of stay (days)", "Total charges ($)")

for (i in seq_along(outcomes)) {
    out <- outcomes[i]
    fmla <- as.formula(paste0("~", out))

    m_psych <- svymean(fmla, svy_surg_psych, na.rm = TRUE)
    m_nopsych <- svymean(fmla, svy_surg_nopsych, na.rm = TRUE)

    if (out %in% c("died", "nonroutine_dc")) {
        cat(sprintf("  %-30s Psych: %.1f%%  No psych: %.1f%%\n",
                    outcome_labels[i],
                    100 * coef(m_psych)[1], 100 * coef(m_nopsych)[1]))
    } else {
        cat(sprintf("  %-30s Psych: %.1f   No psych: %.1f\n",
                    outcome_labels[i],
                    coef(m_psych)[1], coef(m_nopsych)[1]))
    }
}
invisible(NULL)

## Adjusted Models: Effect of Psychiatric Comorbidity on Surgical Outcomes

In [ ]:
# ============================================================
# ADJUSTED MODELS: Among surgery patients
# ============================================================
covariates <- "any_psych + AGE + female + race_f + pay_f + epilepsy_type + surgery_group + YEAR"

# --- Mortality ---
cat("MODEL 1: In-hospital mortality (logistic)\n")
m_mort <- svyglm(as.formula(paste("died ~", covariates)),
                 design = svy_surg, family = quasibinomial())
s_mort <- summary(m_mort)
or_mort <- exp(cbind(OR = coef(m_mort), confint(m_mort)))
cat(sprintf("  any_psych OR: %.2f (%.2f-%.2f), p=%s\n",
            or_mort["any_psych", 1], or_mort["any_psych", 2], or_mort["any_psych", 3],
            ifelse(s_mort$coefficients["any_psych", 4] < 0.001, "<0.001",
                   sprintf("%.3f", s_mort$coefficients["any_psych", 4]))))

# --- Nonroutine discharge ---
cat("\nMODEL 2: Nonroutine discharge (logistic)\n")
m_dc <- svyglm(as.formula(paste("nonroutine_dc ~", covariates)),
               design = svy_surg, family = quasibinomial())
s_dc <- summary(m_dc)
or_dc <- exp(cbind(OR = coef(m_dc), confint(m_dc)))
cat(sprintf("  any_psych OR: %.2f (%.2f-%.2f), p=%s\n",
            or_dc["any_psych", 1], or_dc["any_psych", 2], or_dc["any_psych", 3],
            ifelse(s_dc$coefficients["any_psych", 4] < 0.001, "<0.001",
                   sprintf("%.3f", s_dc$coefficients["any_psych", 4]))))

# --- LOS (log-linear) ---
cat("\nMODEL 3: Length of stay (log-linear)\n")
m_los <- svyglm(as.formula(paste("log_los ~", covariates)),
                design = svy_surg, family = gaussian())
s_los <- summary(m_los)
cat(sprintf("  any_psych coef: %.3f (%.1f%% change in LOS), p=%s\n",
            coef(m_los)["any_psych"],
            100 * (exp(coef(m_los)["any_psych"]) - 1),
            ifelse(s_los$coefficients["any_psych", 4] < 0.001, "<0.001",
                   sprintf("%.3f", s_los$coefficients["any_psych", 4]))))

# --- Total charges (log-linear) ---
cat("\nMODEL 4: Total charges (log-linear)\n")
m_chg <- svyglm(as.formula(paste("log_totchg ~", covariates)),
                design = svy_surg, family = gaussian())
s_chg <- summary(m_chg)
cat(sprintf("  any_psych coef: %.3f (%.1f%% change in charges), p=%s\n",
            coef(m_chg)["any_psych"],
            100 * (exp(coef(m_chg)["any_psych"]) - 1),
            ifelse(s_chg$coefficients["any_psych", 4] < 0.001, "<0.001",
                   sprintf("%.3f", s_chg$coefficients["any_psych", 4]))))

invisible(NULL)

## Disorder-Specific Outcome Models

In [ ]:
# ============================================================
# DISORDER-SPECIFIC OUTCOME MODELS (mortality, among surgery patients)
# ============================================================
psych_labels <- c(
    "psych_depression" = "Depression", "psych_bipolar" = "Bipolar",
    "psych_anxiety" = "Anxiety", "psych_schizophrenia" = "Schizophrenia",
    "psych_alcohol" = "Alcohol", "psych_drug" = "Drug",
    "psych_adhd" = "ADHD", "psych_psychosis" = "Psychosis"
)

base_covars <- "AGE + female + race_f + pay_f + epilepsy_type + surgery_group + YEAR"

cat("DISORDER-SPECIFIC MORTALITY ORs (surgery patients, adjusted)\n\n")
cat(sprintf("%-25s  %8s  %8s  %8s  %10s\n", "Disorder", "OR", "Lower", "Upper", "p-value"))
cat(paste(rep("-", 65), collapse = ""), "\n")

disorder_ors <- data.frame()
for (col in names(psych_labels)) {
    fmla <- as.formula(paste("died ~", col, "+", base_covars))
    m <- tryCatch(svyglm(fmla, design = svy_surg, family = quasibinomial()),
                  error = function(e) NULL)

    if (!is.null(m)) {
        s <- summary(m)
        or_ci <- exp(cbind(OR = coef(m), confint(m)))
        p <- s$coefficients[col, 4]
        p_str <- ifelse(p < 0.001, "<0.001", sprintf("%.3f", p))

        cat(sprintf("%-25s  %8.2f  %8.2f  %8.2f  %10s\n",
                    psych_labels[col], or_ci[col, 1], or_ci[col, 2], or_ci[col, 3], p_str))

        disorder_ors <- rbind(disorder_ors, data.frame(
            Disorder = psych_labels[col],
            Outcome = "Mortality",
            OR = or_ci[col, 1], Lower = or_ci[col, 2], Upper = or_ci[col, 3], P = p
        ))
    }
}

# Also run for LOS
cat("\n\nDISORDER-SPECIFIC LOS EFFECT (surgery patients, adjusted)\n\n")
cat(sprintf("%-25s  %10s  %8s  %8s  %10s\n", "Disorder", "%LOS chg", "Lower", "Upper", "p-value"))
cat(paste(rep("-", 70), collapse = ""), "\n")

for (col in names(psych_labels)) {
    fmla <- as.formula(paste("log_los ~", col, "+", base_covars))
    m <- tryCatch(svyglm(fmla, design = svy_surg, family = gaussian()),
                  error = function(e) NULL)

    if (!is.null(m)) {
        s <- summary(m)
        ci <- confint(m)
        pct_change <- 100 * (exp(coef(m)[col]) - 1)
        pct_lo <- 100 * (exp(ci[col, 1]) - 1)
        pct_hi <- 100 * (exp(ci[col, 2]) - 1)
        p <- s$coefficients[col, 4]
        p_str <- ifelse(p < 0.001, "<0.001", sprintf("%.3f", p))

        cat(sprintf("%-25s  %9.1f%%  %7.1f%%  %7.1f%%  %10s\n",
                    psych_labels[col], pct_change, pct_lo, pct_hi, p_str))

        disorder_ors <- rbind(disorder_ors, data.frame(
            Disorder = psych_labels[col],
            Outcome = "LOS",
            OR = pct_change, Lower = pct_lo, Upper = pct_hi, P = p
        ))
    }
}

write.csv(disorder_ors, file.path(outdir, "tables/disorder_specific_outcomes.csv"), row.names = FALSE)
cat("\nSaved tables/disorder_specific_outcomes.csv\n")
invisible(NULL)

## Figure 6: Forest Plot — Disorder-Specific Mortality ORs in Surgery Patients

In [ ]:
# ============================================================
# FIGURE 6: Forest plot — disorder-specific mortality ORs
# ============================================================
mort_data <- disorder_ors[disorder_ors$Outcome == "Mortality", ]
mort_data$Disorder <- factor(mort_data$Disorder, levels = mort_data$Disorder[order(mort_data$OR)])

p6 <- ggplot(mort_data, aes(x = OR, y = Disorder)) +
    geom_point(size = 3, color = "#E63946") +
    geom_errorbarh(aes(xmin = Lower, xmax = Upper), height = 0.3, color = "#E63946") +
    geom_vline(xintercept = 1, linetype = "dashed", color = "gray40") +
    labs(x = "Adjusted Odds Ratio (95% CI)",
         y = NULL,
         title = "Disorder-Specific Mortality ORs in Epilepsy Surgery Patients",
         subtitle = "NIS 2011-2020, adjusted for age, sex, race, payer, epilepsy type, surgery type, year") +
    theme_minimal(base_size = 12) +
    theme(panel.grid.minor = element_blank())

print(p6)

ggsave(file.path(outdir, "figures/fig6_mortality_ors.pdf"), p6,
       width = 9, height = 6, dpi = 300)
ggsave(file.path(outdir, "figures/fig6_mortality_ors.png"), p6,
       width = 9, height = 6, dpi = 300)
cat("Saved figures/fig6_mortality_ors.pdf/.png\n")
invisible(NULL)

## Fix: Full-Cohort Interaction Models

The surgery-only mortality models above suffer from quasi-separation (very few deaths among ~3,350 surgery patients). Three fixes:

1. **Full-cohort interaction model**: `died ~ psych * surgery + covariates` on all 271K patients
2. **Grouped disorder categories** to increase cell counts
3. **Firth penalized logistic** for the surgery-only models

In [ ]:
# ============================================================
# FIX 1: Full-cohort interaction models
# Uses all 271K patients — tests whether psychiatric comorbidity
# modifies the effect of surgery on outcomes
# ============================================================
cat("FIX 1: FULL-COHORT INTERACTION MODELS\n")
cat("(died ~ any_psych * any_surgery + covariates, N=271,077)\n\n")

dt[, surgery_int := as.integer(any_surgery)]

# Rebuild full survey design with new variable
svy_all <- svydesign(id = ~HOSP_NIS, strata = ~NIS_STRATUM, weights = ~DISCWT,
                     nest = TRUE, data = dt)

base_covars_full <- "AGE + female + race_f + pay_f + epilepsy_type + YEAR"

# Mortality interaction
m_mort_int <- svyglm(
    as.formula(paste("died ~ any_psych * surgery_int +", base_covars_full)),
    design = svy_all, family = quasibinomial()
)
s <- summary(m_mort_int)
or_table <- exp(cbind(OR = coef(m_mort_int), confint(m_mort_int)))

cat("--- Mortality (logistic) ---\n")
for (v in c("any_psych", "surgery_int", "any_psych:surgery_int")) {
    if (v %in% rownames(or_table)) {
        p <- s$coefficients[v, 4]
        p_str <- ifelse(p < 0.001, "<0.001", sprintf("%.3f", p))
        cat(sprintf("  %-30s OR %.2f (%.2f-%.2f), p=%s\n",
                    v, or_table[v, 1], or_table[v, 2], or_table[v, 3], p_str))
    }
}

# Nonroutine discharge interaction
m_dc_int <- svyglm(
    as.formula(paste("nonroutine_dc ~ any_psych * surgery_int +", base_covars_full)),
    design = svy_all, family = quasibinomial()
)
s_dc <- summary(m_dc_int)
or_dc <- exp(cbind(OR = coef(m_dc_int), confint(m_dc_int)))

cat("\n--- Nonroutine discharge (logistic) ---\n")
for (v in c("any_psych", "surgery_int", "any_psych:surgery_int")) {
    if (v %in% rownames(or_dc)) {
        p <- s_dc$coefficients[v, 4]
        p_str <- ifelse(p < 0.001, "<0.001", sprintf("%.3f", p))
        cat(sprintf("  %-30s OR %.2f (%.2f-%.2f), p=%s\n",
                    v, or_dc[v, 1], or_dc[v, 2], or_dc[v, 3], p_str))
    }
}

# LOS interaction
m_los_int <- svyglm(
    as.formula(paste("log_los ~ any_psych * surgery_int +", base_covars_full)),
    design = svy_all, family = gaussian()
)
s_los <- summary(m_los_int)

cat("\n--- Length of stay (log-linear) ---\n")
for (v in c("any_psych", "surgery_int", "any_psych:surgery_int")) {
    if (v %in% names(coef(m_los_int))) {
        p <- s_los$coefficients[v, 4]
        p_str <- ifelse(p < 0.001, "<0.001", sprintf("%.3f", p))
        pct <- 100 * (exp(coef(m_los_int)[v]) - 1)
        cat(sprintf("  %-30s %+.1f%% LOS change, p=%s\n", v, pct, p_str))
    }
}
invisible(NULL)

In [ ]:
# ============================================================
# FIX 2: Grouped disorder categories (surgery-only)
# Combines related disorders to increase cell counts
# ============================================================
cat("FIX 2: GROUPED DISORDER CATEGORIES (surgery patients only)\n\n")

dt_surg[, psych_mood := as.integer(psych_depression == 1 | psych_bipolar == 1)]
dt_surg[, psych_anxiety_spectrum := as.integer(psych_anxiety == 1 | psych_ptsd == 1 | psych_ocd == 1)]
dt_surg[, psych_psychotic := as.integer(psych_schizophrenia == 1 | psych_psychosis == 1)]
dt_surg[, psych_substance := as.integer(psych_alcohol == 1 | psych_drug == 1)]

grouped_labels <- c(
    "psych_mood" = "Mood (depression + bipolar)",
    "psych_anxiety_spectrum" = "Anxiety spectrum (anxiety + PTSD + OCD)",
    "psych_psychotic" = "Psychotic (schizophrenia + psychosis)",
    "psych_substance" = "Substance use (alcohol + drug)"
)

# Prevalence in surgery patients
cat("Grouped prevalence in surgery patients:\n")
for (col in names(grouped_labels)) {
    n <- sum(dt_surg[[col]], na.rm = TRUE)
    cat(sprintf("  %-45s %s (%.1f%%)\n", grouped_labels[col],
                format(n, big.mark = ","), 100 * n / nrow(dt_surg)))
}

# Deaths per group
cat("\nDeaths per group:\n")
for (col in names(grouped_labels)) {
    d <- sum(dt_surg[[col]] == 1 & dt_surg$died == 1, na.rm = TRUE)
    n <- sum(dt_surg[[col]] == 1, na.rm = TRUE)
    cat(sprintf("  %-45s %d / %s (%.1f%%)\n", grouped_labels[col],
                d, format(n, big.mark = ","), ifelse(n > 0, 100 * d / n, 0)))
}

# Rebuild surgery survey design
svy_surg2 <- svydesign(id = ~HOSP_NIS, strata = ~NIS_STRATUM, weights = ~DISCWT,
                       nest = TRUE, data = dt_surg)

base_covars <- "AGE + female + race_f + pay_f + epilepsy_type + surgery_group + YEAR"

# LOS and nonroutine DC models (these work better than mortality)
cat("\n--- Nonroutine discharge ORs (grouped) ---\n")
for (col in names(grouped_labels)) {
    fmla <- as.formula(paste("nonroutine_dc ~", col, "+", base_covars))
    m <- tryCatch(svyglm(fmla, design = svy_surg2, family = quasibinomial()), error = function(e) NULL)
    if (!is.null(m)) {
        s <- summary(m)
        or_ci <- exp(cbind(OR = coef(m), confint(m)))
        p <- s$coefficients[col, 4]
        p_str <- ifelse(p < 0.001, "<0.001", sprintf("%.3f", p))
        cat(sprintf("  %-45s OR %.2f (%.2f-%.2f), p=%s\n",
                    grouped_labels[col], or_ci[col, 1], or_ci[col, 2], or_ci[col, 3], p_str))
    }
}

cat("\n--- LOS % change (grouped) ---\n")
for (col in names(grouped_labels)) {
    fmla <- as.formula(paste("log_los ~", col, "+", base_covars))
    m <- tryCatch(svyglm(fmla, design = svy_surg2, family = gaussian()), error = function(e) NULL)
    if (!is.null(m)) {
        s <- summary(m)
        ci <- confint(m)
        pct <- 100 * (exp(coef(m)[col]) - 1)
        pct_lo <- 100 * (exp(ci[col, 1]) - 1)
        pct_hi <- 100 * (exp(ci[col, 2]) - 1)
        p <- s$coefficients[col, 4]
        p_str <- ifelse(p < 0.001, "<0.001", sprintf("%.3f", p))
        cat(sprintf("  %-45s %+.1f%% (%.1f to %.1f%%), p=%s\n",
                    grouped_labels[col], pct, pct_lo, pct_hi, p_str))
    }
}
invisible(NULL)

In [ ]:
# ============================================================
# FIX 3: Firth penalized logistic regression (surgery-only mortality)
# Handles quasi-separation by adding Jeffrey's prior penalty
# ============================================================
if (!require(logistf, quietly = TRUE)) install.packages("logistf", repos = "https://cran.r-project.org")
library(logistf)

cat("FIX 3: FIRTH PENALIZED LOGISTIC (surgery patients, mortality)\n")
cat("Note: unweighted — Firth logistf does not support survey weights\n\n")

# Prepare unweighted data for Firth
firth_data <- as.data.frame(dt_surg[, .(died, any_psych,
    psych_mood, psych_anxiety_spectrum, psych_psychotic, psych_substance,
    AGE, female, epilepsy_type, surgery_group, YEAR)])
firth_data <- na.omit(firth_data)

cat(sprintf("Firth sample: %s (deaths: %d)\n\n", format(nrow(firth_data), big.mark = ","),
            sum(firth_data$died)))

# Firth: any_psych
m_firth <- logistf(died ~ any_psych + AGE + female + epilepsy_type + surgery_group + YEAR,
                   data = firth_data)
cat("--- Any psychiatric (Firth) ---\n")
idx <- which(names(coef(m_firth)) == "any_psych")
or_f <- exp(coef(m_firth)[idx])
ci_f <- exp(confint(m_firth)[idx, ])
p_f <- m_firth$prob[idx]
cat(sprintf("  any_psych OR: %.2f (%.2f-%.2f), p=%s\n\n",
            or_f, ci_f[1], ci_f[2],
            ifelse(p_f < 0.001, "<0.001", sprintf("%.3f", p_f))))

# Firth: grouped disorders
cat("--- Grouped disorders (Firth) ---\n")
for (col in names(grouped_labels)) {
    fmla <- as.formula(paste("died ~", col, "+ AGE + female + epilepsy_type + surgery_group + YEAR"))
    m_f <- tryCatch(logistf(fmla, data = firth_data), error = function(e) NULL)
    if (!is.null(m_f)) {
        idx <- which(names(coef(m_f)) == col)
        if (length(idx) > 0) {
            or_f <- exp(coef(m_f)[idx])
            ci_f <- exp(confint(m_f)[idx, ])
            p_f <- m_f$prob[idx]
            cat(sprintf("  %-45s OR %.2f (%.2f-%.2f), p=%s\n",
                        grouped_labels[col], or_f, ci_f[1], ci_f[2],
                        ifelse(p_f < 0.001, "<0.001", sprintf("%.3f", p_f))))
        }
    }
}
invisible(NULL)

In [ ]:
# ============================================================
# SUMMARY TABLE: All fixed outcome models
# ============================================================
cat("============================================================\n")
cat("SUMMARY OF OUTCOME ANALYSES\n")
cat("============================================================\n\n")

cat("A. SURGERY-ONLY: any_psych effect (survey-weighted)\n")
cat("   Mortality:        OR 3.95 (0.62-25.37), p=0.147  [wide CI, few events]\n")
cat("   Nonroutine DC:    OR 1.25 (1.03-1.53), p=0.026   ***\n")
cat("   LOS:              +9.5%, p=0.009                   ***\n")
cat("   Charges:          +2.9%, p=0.303\n\n")

cat("B. FULL COHORT: psych * surgery interaction (see above)\n")
cat("   Tests whether psychiatric comorbidity modifies surgical outcomes\n\n")

cat("C. SURGERY-ONLY: Grouped disorders (see above)\n")
cat("   More stable estimates by combining related disorders\n\n")

cat("D. SURGERY-ONLY: Firth penalized logistic (see above)\n")
cat("   Handles quasi-separation in mortality models\n\n")

cat("CONCLUSION: In epilepsy surgery patients, psychiatric comorbidity is\n")
cat("independently associated with longer LOS (+9.5%) and higher rates of\n")
cat("nonroutine discharge (OR 1.25). Mortality effect is underpowered\n")
cat("(very few deaths in the surgery cohort).\n")
invisible(NULL)